# AthleteLoad: 03. Model Evaluation & Benchmark Comparison

This notebook compares Logistic Regression, Random Forest, XGBoost, and LightGBM across temporal validation and test partitions. We also evaluate the secondary objective: next-day athlete readiness score prediction.

In [ ]:
import os
import sys
# Ensure parent directory is in python path
if os.path.abspath('..') not in sys.path:
    sys.path.insert(0, os.path.abspath('..'))

def get_path(rel_path):
    if os.path.exists(rel_path):
        return rel_path
    elif os.path.exists(os.path.join('..', rel_path)):
        return os.path.join('..', rel_path)
    return rel_path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, auc, average_precision_score

from src.utils import setup_plotting_style, PALETTE
from src.train import run_full_training_pipeline
from src.evaluate import run_model_comparison
setup_plotting_style()

artifacts = run_full_training_pipeline(data_path=get_path('data/processed/player_daily_features.csv'), seed=42)
comp_df = run_model_comparison(artifacts)
display(comp_df)

## 1. Test Set ROC & Precision-Recall Curves

In [ ]:
models = artifacts['models']
X_test = artifacts['X_test']
X_test_scaled = artifacts['X_test_scaled']
y_test = artifacts['y_test']

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for name, model in models.items():
    X = X_test_scaled if 'Logistic' in name else X_test
    probs = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_val = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, linewidth=2.0, label=f'{name} ({roc_val:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Guess')
axes[0].set_title('Test Set ROC Curves', fontsize=12, fontweight='bold')
axes[0].set_xlabel('False Positive Rate', fontweight='bold')
axes[0].set_ylabel('True Positive Rate (Recall)', fontweight='bold')
axes[0].legend(loc='lower right', frameon=True)

prevalence = np.mean(y_test)
for name, model in models.items():
    X = X_test_scaled if 'Logistic' in name else X_test
    probs = model.predict_proba(X)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    axes[1].plot(rec, prec, linewidth=2.0, label=f'{name} (PR-AUC={ap:.4f})')
axes[1].axhline(prevalence, color='k', linestyle='--', alpha=0.5, label=f'Prevalence ({prevalence:.2%})')
axes[1].set_title('Test Set Precision-Recall Curves', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Recall (Sensitivity)', fontweight='bold')
axes[1].set_ylabel('Precision (PPV)', fontweight='bold')
axes[1].legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

## 2. Secondary Task: Next-Day Readiness Prediction

In [ ]:
r_model = artifacts['readiness_models']['XGBoost Regressor']
val_df = artifacts['val_df']
feature_cols = artifacts['feature_cols']
imputer = artifacts['imputer']
X_val = imputer.transform(val_df[feature_cols].values)
y_val_r = val_df['readiness_next_day'].values

val_mask = ~np.isnan(y_val_r)
y_true_v = y_val_r[val_mask]
y_pred_v = r_model.predict(X_val[val_mask])

plt.figure(figsize=(7, 6))
sns.regplot(x=y_true_v, y=y_pred_v, scatter_kws={'alpha': 0.25, 'color': PALETTE['primary']}, line_kws={'color': PALETTE['danger'], 'linewidth': 2.0})
plt.plot([1, 10], [1, 10], 'k--', alpha=0.6, label='Ideal 1:1 Reference')
plt.title('Next-Day Athlete Readiness Prediction (XGBoost Regressor)', fontsize=12, fontweight='bold')
plt.xlabel('Actual Next-Day Readiness (1-10)', fontweight='bold')
plt.ylabel('Predicted Readiness Score', fontweight='bold')
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()